# Ces scripts servent à obtenir et ranger les différents infinitifs des auteurs

##  <span style="color: #FFFFE0; font-weight: bold;">Conversion des textes format word en format txt</span>

In [10]:
# Ce script convertit les textes des auteurs en format .txt si ce n'est pas déjà le cas dans un but d'uniformisation et de faciliation des prochains scripts

import os
import win32com.client
import shutil
import subprocess

def convert_doc_to_txt(doc_path, txt_path):
    try:
        # Créer une instance de l'application Word
        word = win32com.client.Dispatch("Word.Application")
        word.Visible = False

        # Ouvrir le document .doc ou .docx
        doc = word.Documents.Open(doc_path)

        # Enregistrer le contenu sous forme de texte brut
        doc.SaveAs(txt_path, FileFormat=2)  # 2 correspond au format .txt
    except Exception as e:
        print(f"Erreur lors de la conversion de {doc_path}: {e}")
    finally:
        try:
            # Fermer le document et quitter Word
            doc.Close(False)
            word.Quit()
        except Exception as e:
            print(f"Erreur lors de la fermeture du document ou de Word: {e}")

def convert_odt_to_txt(odt_path, txt_path):
    try:
        # Utiliser LibreOffice pour convertir le fichier .odt en .txt
        subprocess.run([
            'libreoffice', '--headless', '--convert-to', 'txt', '--outdir',
            os.path.dirname(txt_path), odt_path
        ])
    except Exception as e:
        print(f"Erreur lors de la conversion de {odt_path}: {e}")

def convert_all_docs_to_txt(input_folder, output_folder):
    # Vérifier si le dossier d'entrée existe
    if not os.path.exists(input_folder):
        print(f"Le dossier d'entrée n'existe pas: {input_folder}")
        return

    # Créer le dossier de sortie s'il n'existe pas
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Parcourir tous les fichiers dans le dossier d'entrée
    for filename in os.listdir(input_folder):
        input_path = os.path.join(input_folder, filename)
        txt_filename = os.path.splitext(filename)[0] + '.txt'
        txt_path = os.path.join(output_folder, txt_filename)

        if filename.endswith('.odt'):
            if not os.path.exists(txt_path):
                convert_odt_to_txt(input_path, txt_path)
                print(f"Converti : {filename} -> {txt_filename}")
        elif filename.endswith(('.doc', '.docx')):
            if not os.path.exists(txt_path):
                convert_doc_to_txt(input_path, txt_path)
                print(f"Converti : {filename} -> {txt_filename}")
        elif filename.endswith('.txt'):
            # Copier les fichiers .txt directement s'ils sont déjà en format .txt
            if not os.path.exists(txt_path):
                shutil.copy(input_path, output_folder)
                print(f"Copié : {filename}")

# Chemins des dossiers pour Poitevin
poitevin_input_folder = r'C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Textes bruts'
poitevin_output_folder = r'C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Textes en txt'

# Chemins des dossiers pour Saintongeais
saintongeais_input_folder = r'C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Textes bruts'
saintongeais_output_folder = r'C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Textes en txt'

# Convertir tous les fichiers pour les deux dossiers
convert_all_docs_to_txt(poitevin_input_folder, poitevin_output_folder)
convert_all_docs_to_txt(saintongeais_input_folder, saintongeais_output_folder)


##  <span style="color: #FFFFE0; font-weight: bold;">Création d'une liste des terminaisons pour chaque auteur</span>

In [1]:
import os

def process_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.readlines()

    terminations_in_order = []
    for line in content:
        if line.strip().startswith('* 1'):
            parts = line.split(':')
            if len(parts) > 1:
                term_list = parts[1].strip().split(',')
                for term in term_list:
                    term = term.strip()
                    if term != 'X' and term not in terminations_in_order:
                        terminations_in_order.append(term)

    new_line = "\nterm=" + ",".join(terminations_in_order) + "\n"

    # Supprimer l'ancienne ligne "term=" si elle existe
    content = [line for line in content if not line.strip().startswith('term=')]

    # Ajouter la nouvelle ligne à la fin du fichier
    content.append(new_line)

    with open(file_path, 'w', encoding='utf-8') as file:
        file.writelines(content)

def process_directory(directory_path):
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(directory_path, filename)
            process_file(file_path)

# Ne traiter que le dossier "Terminaisons"
directories = [
    r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Localiser locuteurs\Terminaisons"
]

for directory_path in directories:
    process_directory(directory_path)

print("Fini :)")

Fini :)


## <span style="color: #FFFFE0; font-weight: bold;">Obtention des listes d'infinitifs et des correspondances</span>

In [2]:
import os
import re
import pandas as pd
import shutil
from collections import defaultdict
import unicodedata

def lire_terminaisons(fichier):
    with open(fichier, 'r', encoding='utf-8') as file:
        terminaisons = []
        infinitifs = []
        for line in file:
            if line.startswith('term='):
                terminaisons = line.split('=')[1].strip().split(',')
            elif line.startswith('* 1inf :'):
                infinitif = line.split(':')[1].strip()
                infinitifs.append(infinitif)
        return [term.strip() for term in terminaisons], [inf.strip() for inf in infinitifs]

def supprimer_accents(mot):
    nfkd_form = unicodedata.normalize('NFKD', mot)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

def remplacer_apostrophes(mot):
    mot = re.sub(r"c'm", "com", mot)
    mot = re.sub(r"['’]", "e", mot)
    return mot

def contient_chiffres(mot):
    return any(char.isdigit() for char in mot)

def rechercher_mots_par_terminaison(texte, terminaisons, infinitifs):
    terminaisons_triees = sorted(terminaisons, key=len, reverse=True)
    mots_trouves = set()

    mots_interdits = {
        "père", "pères", "mère", "mères", "crème", "crèmes", "fille", "filles",
        "fesant", "fasse", "femme", "femmes", "feme", "femes", "seul", "seule",
        "seuls", "seules", "toute", "toutes", "route", "routes", "belle", "belles"
    }

    with open(texte, 'r', encoding='cp1252') as file:
        for line in file:
            mots = line.split()
            for mot in mots:
                mot_nettoye = mot.lower()
                mot_nettoye = re.sub(r'[()\[\]":«».!?#,_=;—]', '', mot_nettoye)
                if '-' in mot_nettoye:
                    continue
                if "'" in mot_nettoye[:3] or "’" in mot_nettoye[:3] or "‘" in mot_nettoye[:3]:
                    if "'" in mot_nettoye:
                        mot_nettoye = mot_nettoye[mot_nettoye.index("'") + 1:]
                    elif "’" in mot_nettoye:
                        mot_nettoye = mot_nettoye[mot_nettoye.index("’") + 1:]
                    elif "‘" in mot_nettoye:
                        mot_nettoye = mot_nettoye[mot_nettoye.index("‘") + 1:]

                if len(mot_nettoye) >= 4 and mot_nettoye not in mots_interdits:
                    mot_trouve = False
                    for term in terminaisons_triees:
                        if mot_nettoye.endswith(term):
                            base_verbale = mot_nettoye[:-len(term)]
                            # Exclure les radicaux "cop" et "pllaç"
                            if base_verbale.startswith("cop") or base_verbale.startswith("pllaç"):
                                continue
                            for infinitif in infinitifs:
                                mot_avec_infinitif = f"{base_verbale}{infinitif}"
                                mots_trouves.add(mot_avec_infinitif)
                            mot_trouve = True
                            break
                    if not mot_trouve:
                        for term in terminaisons_triees:
                            term_modifie = term.replace('e', "'")
                            if mot_nettoye.endswith(term_modifie):
                                base_verbale = mot_nettoye[:-len(term_modifie)]
                                # Exclure les radicaux "cop" et "pllaç"
                                if base_verbale.startswith("cop") or base_verbale.startswith("pllaç"):
                                    continue
                                for infinitif in infinitifs:
                                    mot_avec_infinitif = f"{base_verbale}{infinitif}"
                                    mots_trouves.add(mot_avec_infinitif)
                                break

    mots_trouves = {remplacer_apostrophes(mot) for mot in mots_trouves if not contient_chiffres(mot)}
    return mots_trouves

def lire_dicopoitevin(fichier):
    df = pd.read_csv(fichier, sep='\t', encoding='utf-8')
    return pd.Series(df['forme2'].values, index=df['forme']).to_dict()

def sauvegarder_resultats(mots_trouves, fichier_sortie, infinitifs):
    os.makedirs(os.path.dirname(fichier_sortie), exist_ok=True)
    with open(fichier_sortie, 'w', encoding='utf-8') as file:
        file.write(f"Inf: {', '.join(infinitifs)}\n")
        for mot in sorted(mots_trouves):
            file.write(f"{mot}\n")

def sauvegarder_correspondances(mots_correspondances, fichier_sortie, mots_dicopoitevin):
    fichier_sortie = fichier_sortie.replace('.txt', '_O.txt')
    os.makedirs(os.path.dirname(fichier_sortie), exist_ok=True)
    with open(fichier_sortie, 'w', encoding='utf-8') as file:
        file.write("Auteur\tDicopoitevin\n")
        for mot_auteur, mot_dico in sorted(mots_correspondances):
            if len(mot_auteur) >= 4:
                mot_forme2 = mots_dicopoitevin.get(mot_dico, mot_dico)
                if mot_forme2.endswith("ér"):
                    mot_forme2 = mot_forme2[:-2] + "àe"
                file.write(f"{mot_auteur}\t{mot_forme2}\n")

def sauvegarder_non_correspondances(mots_non_correspondances, fichier_sortie):
    fichier_sortie = fichier_sortie.replace('.txt', '_X.txt')
    os.makedirs(os.path.dirname(fichier_sortie), exist_ok=True)
    with open(fichier_sortie, 'w', encoding='utf-8') as file:
        for mot in sorted(mots_non_correspondances):
            file.write(f"{mot}\n")

def copier_et_compter_correspondances(dossier_correspondances, dossier_destination):
    os.makedirs(dossier_destination, exist_ok=True)
    total_lignes = 0

    for fichier in os.listdir(dossier_correspondances):
        if fichier.endswith('_O.txt'):
            chemin_source = os.path.join(dossier_correspondances, fichier)
            chemin_destination = os.path.join(dossier_destination, fichier.replace('_O.txt', '.txt'))
            shutil.copy(chemin_source, chemin_destination)

            with open(chemin_destination, 'r', encoding='utf-8') as file:
                lignes = file.readlines()
                lignes_a_compter = lignes[1:]
                if lignes_a_compter and lignes_a_compter[-1].strip() == '':
                    lignes_a_compter = lignes_a_compter[:-1]
                total_lignes += len(lignes_a_compter)

    return total_lignes

def correspondance_avec_particularites(base_auteur, base_dico):
    if any(x in base_auteur or x in base_dico for x in ["ouss", "os", "auch", "auss", "ten", "tan", "cont", "cent", "ajhou", "agou"]):
        return False
    if base_auteur == base_dico:
        return True
    if base_auteur.replace('qu', 'c') == base_dico or base_auteur.replace('c', 'qu') == base_dico:
        return True
    if base_auteur.replace('zin', 'sin') == base_dico:
        return True
    if base_auteur.replace('an', 'en') == base_dico and not ('van' in base_auteur or 'ven' in base_dico):
        return True
    if base_auteur.replace('en', 'an') == base_dico and not ('ven' in base_auteur or 'van' in base_dico):
        return True
    if base_auteur.replace('aim', 'aem') == base_dico or base_auteur.replace('aem', 'aim') == base_dico:
        return True
    if base_auteur.replace('emp', 'enp') == base_dico or base_auteur.replace('enp', 'emp') == base_dico:
        return True
    if base_auteur.replace('emt', 'ent') == base_dico or base_auteur.replace('ent', 'emt') == base_dico:
        return True
    if base_auteur.replace('emf', 'enf') == base_dico or base_auteur.replace('enf', 'emf') == base_dico:
        return True
    if base_auteur.replace('ou', 'o') == base_dico and not base_dico.endswith('t'):
        return True
    if base_auteur.replace('o', 'ou') == base_dico and not base_dico.endswith('t'):
        return True
    if base_auteur.replace('au', 'o') == base_dico or base_auteur.replace('o', 'au') == base_dico:
        return True
    if base_auteur.replace('eu', 'e') == base_dico or base_auteur.replace('e', 'eu') == base_dico:
        return True
    if base_auteur.replace('ein', 'ain') == base_dico or base_auteur.replace('ain', 'ein') == base_dico:
        return True
    if base_auteur.replace('jh', 'g') == base_dico or base_auteur.replace('g', 'jh') == base_dico:
        return True
    if base_auteur.replace('jh', 'j') == base_dico or base_auteur.replace('j', 'jh') == base_dico:
        return True
    if base_auteur.replace('gh', 'jh') == base_dico or base_auteur.replace('g', 'gh') == base_dico:
        return True
    if base_auteur.replace('ns', 'nz') == base_dico or base_auteur.replace('nz', 'ns') == base_dico:
        return True
    if base_auteur.replace('n', 'nn') == base_dico or base_auteur.replace('nn', 'n') == base_dico:
        return True
    if base_auteur.replace('l', 'll') == base_dico or base_auteur.replace('ll', 'l') == base_dico:
        return True
    if base_auteur.replace('s', 'ss') == base_dico or base_auteur.replace('ss', 's') == base_dico:
        return True
    if base_auteur.replace('m', 'mm') == base_dico or base_auteur.replace('mm', 'm') == base_dico:
        return True
    if base_auteur.replace('r', 'rr') == base_dico or base_auteur.replace('rr', 'r') == base_dico:
        return True
    if base_auteur.replace('t', 'tt') == base_dico or base_auteur.replace('tt', 't') == base_dico:
        return True
    if base_auteur.replace('c', 'k') == base_dico or base_auteur.replace('k', 'c') == base_dico:
        return True
    if base_auteur.replace('pp', 'p') == base_dico or base_auteur.replace('p', 'pp') == base_dico:
        return True
    if base_auteur.replace('ff', 'f') == base_dico or base_auteur.replace('f', 'ff') == base_dico:
        return True
    if base_auteur.replace('cc', 'c') == base_dico or base_auteur.replace('c', 'cc') == base_dico:
        return True
    if base_auteur.replace('dj', 'd') == base_dico or base_auteur.replace('d', 'dj') == base_dico:
        return True
    if base_auteur.replace('gg', 'g') == base_dico or base_auteur.replace('g', 'gg') == base_dico:
        return True
    if base_auteur.replace('bb', 'b') == base_dico or base_auteur.replace('b', 'bb') == base_dico:
        return True
    if base_auteur.replace('gua', 'ga') == base_dico or base_auteur.replace('ga', 'gua') == base_dico:
        return True
    if base_auteur.replace('dem', 'dm') == base_dico or base_auteur.replace('dm', 'dem') == base_dico:
        return True
    if base_auteur.replace('ph', 'f') == base_dico:
        return True
    if base_auteur.replace('ill', 'y') == base_dico:
        return True
    if base_auteur.replace('cher', 'char') == base_dico:
        return True
    if base_auteur.replace('jh', 'y') == base_dico:
        return True
    if base_auteur.replace('com', 'cun') == base_dico or base_auteur.replace('con', 'cun') == base_dico:
        return True
    if base_auteur.replace('cant', 'cunt') == base_dico:
        return True
    if base_auteur.replace('ch', 'che') == base_dico:
        return True
    if base_auteur.replace('ail', 'all') == base_dico:
        return True
    if base_auteur.replace('fe', 'fae') == base_dico:
        return True
    if base_auteur.replace('com', 'coum') == base_dico:
        return True
    if base_auteur.replace('gagn', 'gangn') == base_dico:
        return True
    if base_auteur.replace('ign', 'égn') == base_dico:
        return True
    if base_auteur.replace('alon', 'alun') == base_dico:
        return True
    if base_auteur.replace('alig', 'aleg') == base_dico:
        return True
    if base_auteur.replace('me', 'mae') == base_dico or base_auteur.replace('ma', 'mae') == base_dico:
        return True
    if base_auteur.replace('ble', 'blae') == base_dico:
        return True
    if base_auteur.replace('la', 'lae') == base_dico:
        return True
    if base_auteur.replace('pre', 'prae') == base_dico:
        return True
    if base_auteur.replace('aid', 'aed') == base_dico:
        return True
    if base_auteur.replace('lai', 'lae') == base_dico:
        return True
    if base_auteur.replace('lon', 'len') == base_dico:
        return True
    if base_auteur.replace('voy', 'vey') == base_dico:
        return True
    if base_auteur.replace('gon', 'gun') == base_dico:
        return True
    if base_auteur.replace('lig', 'leg') == base_dico:
        return True
    if base_auteur.replace('fon', 'fun') == base_dico:
        return True
    if base_auteur.replace('tax', 'taux') == base_dico:
        return True
    if base_auteur.replace('savo', 'sav') == base_dico:
        return True
    if base_auteur.replace('ferm', 'frm') == base_dico:
        return True
    if base_auteur.replace('éveill', 'évell') == base_dico:
        return True
    if base_auteur.replace('mouill', 'moull') == base_dico:
        return True
    if base_auteur.replace('ennuy', 'enneùy') == base_dico:
        return True
    if base_auteur.replace('coummin', 'quemen') == base_dico or base_auteur.replace('coumoin', 'quemen') == base_dico:
        return True
    if base_auteur.replace('confess', 'cunfae') == base_dico:
        return True
    if base_auteur.replace('conseill', 'cunsel') == base_dico:
        return True
    if base_auteur.replace('compt', 'cunt') == base_dico or base_auteur.replace('cont', 'cunt') == base_dico:
        return True
    if base_auteur.replace('annon', 'anun') == base_dico:
        return True
    if base_auteur.replace('sonn', 'soun') == base_dico:
        return True
    if base_auteur.replace('emagi', 'emajhi') == base_dico:
        return True
    if base_auteur.replace('donn', 'doun') == base_dico:
        return True
    if base_auteur.replace('regard', 'argard') == base_dico:
        return True
    if base_auteur.replace('inquiet', 'inche') == base_dico:
        return True
    if base_auteur.replace('mang', 'menjh') == base_dico:
        return True
    if base_auteur.replace('appell', 'apel') == base_dico:
        return True
    if base_auteur.replace('arret', 'araet') == base_dico:
        return True
    if base_auteur.replace('ennoy', 'ennu') == base_dico:
        return True
    if base_auteur.replace('mon', 'man') == base_dico and not ('mond' in base_auteur or 'mand' in base_dico):
        return True
    if base_auteur.replace('beunais', 'benaes') == base_dico:
        return True
    if base_auteur.replace('comm', 'coum') == base_dico:
        return True

    return False

def traiter_dossier(dossier_terminaisons, dossier_textes, dossier_sortie, dicopoitevin_path, dossier_comptage):
    mots_dicopoitevin = lire_dicopoitevin(dicopoitevin_path)

    fichiers_par_auteur = defaultdict(list)
    for fichier in os.listdir(dossier_textes):
        if fichier.endswith('.txt'):
            match = re.match(r"(.*?)(\s\d+)?\.txt", fichier)
            if match:
                auteur = match.group(1)
                fichiers_par_auteur[auteur].append(fichier)

    for auteur, fichiers in fichiers_par_auteur.items():
        terminaisons, infinitifs = lire_terminaisons(os.path.join(dossier_terminaisons, f"{auteur}.txt"))
        mots_trouves = set()

        for fichier in fichiers:
            texte = os.path.join(dossier_textes, fichier)
            if os.path.exists(texte):
                mots_trouves.update(rechercher_mots_par_terminaison(texte, terminaisons, infinitifs))

        fichier_sortie = os.path.join(dossier_sortie, f"{auteur}.txt")
        sauvegarder_resultats(mots_trouves, fichier_sortie, infinitifs)

        mots_correspondances = []
        mots_non_correspondances = []

        for mot in mots_trouves:
            base_verbale = re.sub(f'({"|".join(infinitifs)})$', '', mot)
            base_sans_accents = supprimer_accents(base_verbale)

            correspondance_trouvee = False
            for dico_mot in mots_dicopoitevin:
                base_dico_sans_accents = supprimer_accents(dico_mot)
                if correspondance_avec_particularites(base_sans_accents, base_dico_sans_accents):
                    mots_correspondances.append((mot, dico_mot))
                    correspondance_trouvee = True
                    break
            if not correspondance_trouvee:
                mots_non_correspondances.append(mot)

        fichier_correspondances = os.path.join(dossier_sortie, 'Correspondances', f"{auteur}.txt")
        sauvegarder_correspondances(mots_correspondances, fichier_correspondances, mots_dicopoitevin)

        fichier_non_correspondances = os.path.join(dossier_sortie, 'Non-correspondances', f"{auteur}.txt")
        sauvegarder_non_correspondances(mots_non_correspondances, fichier_non_correspondances)

    total_lignes = copier_et_compter_correspondances(os.path.join(dossier_sortie, 'Correspondances'), dossier_comptage)

# Chemins des dossiers
terminaisons_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Localiser locuteurs\Terminaisons"
textes_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Textes en txt"
sortie_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Formes trouvées"
comptage_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Formes trouvées\Correspondances comptage"

terminaisons_saintongeais = terminaisons_poitevin
textes_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Textes en txt"
sortie_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Formes trouvées"
comptage_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Formes trouvées\Correspondances comptage"

dicopoitevin_path = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Dicopoitevin\dicopoitevin adapté.tsv"

traiter_dossier(terminaisons_poitevin, textes_poitevin, sortie_poitevin, dicopoitevin_path, comptage_poitevin)
traiter_dossier(terminaisons_saintongeais, textes_saintongeais, sortie_saintongeais, dicopoitevin_path, comptage_saintongeais)

print("Fini :)")


Fini :)


## <span style="color: #FFFFE0; font-weight: bold;">Comptage du nombre de nouveaux verbes à fléchir</span>

In [3]:
import os

def compter_infinitifs_txt(dossier):
    total_infinitifs = 0

    for fichier in os.listdir(dossier):
        if fichier.endswith('.txt'):
            chemin_fichier = os.path.join(dossier, fichier)
            with open(chemin_fichier, 'r', encoding='utf-8') as file:
                lignes = file.readlines()
                # Ne pas compter la première ligne et la dernière si elle est vide
                lignes_a_compter = lignes[1:]
                if lignes_a_compter and lignes_a_compter[-1].strip() == '':
                    lignes_a_compter = lignes_a_compter[:-1]
                total_infinitifs += len(lignes_a_compter)

    return total_infinitifs

def compter_infinitifs_tsv(dossier):
    total_infinitifs = 0

    for fichier in os.listdir(dossier):
        if fichier.endswith('.tsv'):
            chemin_fichier = os.path.join(dossier, fichier)
            with open(chemin_fichier, 'r', encoding='utf-8') as file:
                lignes = file.readlines()
                # Ne pas compter la première ligne (en-tête) et la dernière si elle est vide
                lignes_a_compter = lignes[1:]
                if lignes_a_compter and lignes_a_compter[-1].strip() == '':
                    lignes_a_compter = lignes_a_compter[:-1]
                total_infinitifs += len(lignes_a_compter)

    return total_infinitifs

# Chemins des dossiers
comptage_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Formes trouvées\Correspondances comptage"
comptage_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Formes trouvées\Correspondances comptage"
lexiques_nettoyes_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Lexique locuteur\Lexiques nettoyés"
lexiques_nettoyes_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Lexique locuteur\Lexiques nettoyés"

# Compter les infinitifs
total_poitevin = compter_infinitifs_txt(comptage_poitevin) + compter_infinitifs_tsv(lexiques_nettoyes_poitevin)
total_saintongeais = compter_infinitifs_txt(comptage_saintongeais) + compter_infinitifs_tsv(lexiques_nettoyes_saintongeais)
total_ensemble = total_poitevin + total_saintongeais

# Afficher les résultats
print(f"Infinitifs en poitevin: {total_poitevin}")
print(f"Infinitifs en saintongeais: {total_saintongeais}")
print(f"Total : {total_ensemble}")


Infinitifs en poitevin: 23885
Infinitifs en saintongeais: 9221
Total : 33106


## <span style="color: #FFFFE0; font-weight: bold;">Comptage du nombre de formes à générer</span>

In [6]:
import re
import os

def count_terminations(termination_file_path):
    """Compte le nombre de terminaisons dans le fichier donné, en ignorant 'X'."""
    with open(termination_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        # Trouver toutes les lignes qui commencent par "* 1" et extraire les terminaisons
        termination_lines = re.findall(r'\* 1\w* : ([^\n]*)', content)

        # Compter toutes les terminaisons, en ignorant "X"
        terminations_count = 0
        for line in termination_lines:
            terms = [term.strip() for term in line.split(',') if term.strip() != "X"]
            terminations_count += len(terms)

        return terminations_count

def count_valid_lines_txt(correspondance_file_path):
    """Compte le nombre de lignes valides dans le fichier de correspondance TXT, en ignorant la première ligne et la dernière si elle est vide."""
    with open(correspondance_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    # Ignorer la première ligne et la dernière ligne si elle est vide
    valid_lines = [line for line in lines[1:] if line.strip()]

    return len(valid_lines)

def count_valid_lines_tsv(correspondance_file_path):
    """Compte le nombre de lignes valides dans le fichier de correspondance TSV, en ignorant la première ligne et la dernière si elle est vide."""
    with open(correspondance_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    # Ignorer la première ligne (en-tête) et la dernière ligne si elle est vide
    valid_lines = [line for line in lines[1:] if line.strip()]

    return len(valid_lines)

def calculate_conjugations(terminations_dir, correspondance_dir_txt, correspondance_dir_tsv):
    """Calcule le nombre de conjugaisons possibles pour chaque fichier de terminaison."""
    conjugation_counts = {}

    # Obtenir une liste de tous les fichiers de terminaison
    termination_files = [f for f in os.listdir(terminations_dir) if f.endswith('.txt')]

    for termination_file in termination_files:
        base_file_name = os.path.splitext(termination_file)[0]

        # Trouver les fichiers de correspondance TXT et TSV correspondants
        corresponding_files_txt = [f for f in os.listdir(correspondance_dir_txt) if f.startswith(base_file_name) and f.endswith('.txt')]
        corresponding_files_tsv = [f for f in os.listdir(correspondance_dir_tsv) if f.startswith(base_file_name) and f.endswith('.tsv')]

        termination_file_path = os.path.join(terminations_dir, termination_file)

        # Compter les terminaisons
        num_terminations = count_terminations(termination_file_path)

        # Calculer les conjugaisons pour les fichiers TXT
        for correspondance_file in corresponding_files_txt:
            correspondance_file_path = os.path.join(correspondance_dir_txt, correspondance_file)
            num_valid_lines = count_valid_lines_txt(correspondance_file_path)
            conjugations = num_terminations * num_valid_lines
            conjugation_counts[correspondance_file] = conjugations

        # Calculer les conjugaisons pour les fichiers TSV
        for correspondance_file in corresponding_files_tsv:
            correspondance_file_path = os.path.join(correspondance_dir_tsv, correspondance_file)
            num_valid_lines = count_valid_lines_tsv(correspondance_file_path)
            conjugations = num_terminations * num_valid_lines
            conjugation_counts[correspondance_file] = conjugations

    return conjugation_counts

# Dossiers
terminations_dir = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Localiser locuteurs\Terminaisons"
correspondance_dir_poitevin_txt = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Formes trouvées\Correspondances comptage"
correspondance_dir_saintongeais_txt = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Formes trouvées\Correspondances comptage"
correspondance_dir_poitevin_tsv = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Lexique locuteur\Lexiques nettoyés"
correspondance_dir_saintongeais_tsv = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Lexique locuteur\Lexiques nettoyés"

# Calculer les conjugaisons pour le Poitevin
conjugation_counts_poitevin = calculate_conjugations(terminations_dir, correspondance_dir_poitevin_txt, correspondance_dir_poitevin_tsv)

# Calculer les conjugaisons pour le Saintongeais
conjugation_counts_saintongeais = calculate_conjugations(terminations_dir, correspondance_dir_saintongeais_txt, correspondance_dir_saintongeais_tsv)

# Afficher le résultat total pour le Poitevin
total_conjugations_poitevin = sum(conjugation_counts_poitevin.values())
print(f"Nombre total de conjugaisons possibles (Poitevin) : {total_conjugations_poitevin}")

# Afficher le résultat total pour le Saintongeais
total_conjugations_saintongeais = sum(conjugation_counts_saintongeais.values())
print(f"Nombre total de conjugaisons possibles (Saintongeais) : {total_conjugations_saintongeais}")

# Calculer et afficher le total général
total_conjugations_overall = total_conjugations_poitevin + total_conjugations_saintongeais
print(f"\nNombre total de conjugaisons possibles (Poitevin + Saintongeais) : {total_conjugations_overall}")

Nombre total de conjugaisons possibles (Poitevin) : 988571
Nombre total de conjugaisons possibles (Saintongeais) : 370091

Nombre total de conjugaisons possibles (Poitevin + Saintongeais) : 1358662
